# Homework 3 — Colab starter

Complete the assigned preparation before beginning.

## How to work in this notebook

This `.ipynb` is your **only working document**. The assignment PDF is a read-only copy of the same prompt. Do not edit or combine a `.qmd` file.

- Write reasoning in the designated text cells.
- Run or modify the starter code rather than pasting an unexplained replacement.
- Keep requested output, plots, excerpts, and raw AI evidence visible.
- Handwriting is welcome but never required. Typed Markdown/LaTeX is fully equivalent.
- If you insert a clear scan/photo, add one typed description or statistical conclusion for accessibility.

Before submission, restart and run all. Upload a PDF export and the completed `.ipynb` to the same Gradescope assignment. If image insertion fails, add one clearly labeled optional handwriting PDF; do not merge files.

In [ ]:
# Standard Colab setup - run once
from pathlib import Path
import hashlib
import json
import random
import sys
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 2027
random.seed(SEED)
np.random.seed(SEED)

COURSE_REPO_RAW_URL = 'https://raw.githubusercontent.com/skgallagher/stat-methods-ai-public/main'
COURSE_DATA_BASE_URL = COURSE_REPO_RAW_URL + '/data/course'
COURSE_DATA_GROUPS = ['cfpb', 'github_issues']
RUNTIME_ROOT = Path('/content') if Path('/content').exists() else Path.cwd()
DATA_ROOT = RUNTIME_ROOT / 'stat_ai_data'

# A local notebook may start in the repository root or a nested week folder.
# Search upward for frozen course data before falling back to a download.
search_roots = [Path.cwd(), *Path.cwd().parents]
LOCAL_RELEASE = next((
    root / 'data' / 'course'
    for root in search_roots
    if (root / 'data' / 'course' / 'manifest.json').exists()
), None)
LOCAL_SMOKE = next((
    root / 'data' / 'smoke'
    for root in search_roots
    if (root / 'data' / 'smoke').exists()
), None)
online_release = False
if LOCAL_RELEASE is not None:
    DATA_ROOT = LOCAL_RELEASE
    data_source = 'local frozen release'
elif LOCAL_SMOKE is not None:
    DATA_ROOT = LOCAL_SMOKE
    data_source = 'local synthetic smoke fixture (development only)'
elif (DATA_ROOT / 'manifest.json').exists():
    cached_manifest = json.loads((DATA_ROOT / 'manifest.json').read_text())
    if 'smoke fixture' in cached_manifest.get('bundle_type', ''):
        data_source = 'existing local synthetic smoke fixture (development only)'
    else:
        cached_files = cached_manifest.get('files', [])
        requested_files = [
            item for item in cached_files
            if Path(item['path']).parts[0] in COURSE_DATA_GROUPS
        ]
        def cached_sha256(path):
            digest = hashlib.sha256()
            with path.open('rb') as stream:
                for chunk in iter(lambda: stream.read(1024 * 1024), b''):
                    digest.update(chunk)
            return digest.hexdigest()
        cache_complete = (
            cached_manifest.get('release_status') == 'student_release'
            and requested_files
            and all(
                (DATA_ROOT / item['path']).exists()
                and cached_sha256(DATA_ROOT / item['path']) == item['sha256']
                for item in requested_files
            )
        )
        if cache_complete:
            data_source = 'existing verified runtime cache'
        else:
            online_release = True
else:
    online_release = True

if online_release:
    # Build the package directory as path components so notebook/Markdown
    # converters cannot turn the underscore into a literal escaped path.
    helper_package = 'course' + '_helpers'
    helper_target = RUNTIME_ROOT / helper_package / '__init__.py'
    helper_target.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(
        COURSE_REPO_RAW_URL + '/course_helpers/__init__.py', helper_target
    )
    runtime_import_root = str(RUNTIME_ROOT)
    if runtime_import_root not in sys.path:
        sys.path.insert(0, runtime_import_root)
    from course_helpers import ensure_course_data
    DATA_ROOT = ensure_course_data(
        COURSE_DATA_BASE_URL,
        DATA_ROOT,
        groups=COURSE_DATA_GROUPS,
    )
    data_source = 'public GitHub student release'

print('Setup complete. Data root:', DATA_ROOT)
print('Data source:', data_source)
print('Requested groups:', COURSE_DATA_GROUPS)


| | |
|---|---|
| **Out / due** | Tue Feb 2 / Tue Feb 9, 11:59pm |
| **Points** | 100: Problem 1 (50), Problem 2 (30), Problem 3 (20) |
| **Expected time** | About 5 hours after completing Lab 3; contact course staff if setup/debugging alone exceeds 30 minutes |
| **Reading** | Mitchell et al., [“Model Cards for Model Reporting”](https://arxiv.org/abs/1810.03993), Section 4, especially intended use, metrics, evaluation data, and caveats |
| **AI policy** | No AI for Problem 1; allowed in Problem 2 and required in Problem 3. Submit the AI-use record. |
| **Working file** | Complete all work in `hw03_starter.ipynb`; the PDF is a read-only prompt copy. |
| **Submit** | Completed notebook PDF + completed `.ipynb` in one Gradescope submission. |

In [ ]:
# HW3: inspect the design before fitting any model
issues_root = DATA_ROOT / 'github_issues'
github_issues = pd.read_csv(issues_root / 'issues.csv', parse_dates=['created_date'])
assert github_issues['issue_number'].is_unique
assert len(github_issues) == 120
print('Real scikit-learn issue-triage extract:')
display(github_issues.head(3))
display(github_issues['module_label'].value_counts().rename('n').to_frame())
print('Issues by year and module (use this EDA when planning the split):')
display(pd.crosstab(github_issues['created_year'], github_issues['module_label']))
print('Date range:', github_issues['created_date'].min().date(),
      'to', github_issues['created_date'].max().date())
print('Unique reporter groups:', github_issues['reporter_group'].nunique())
print('The balanced extract does not preserve natural module prevalence.')

# Problem 2 returns to the CFPB data from Lab 3.
cfpb_root = DATA_ROOT / 'cfpb'
complaints = pd.read_csv(cfpb_root / 'complaints.csv')
complaints['date_received'] = pd.to_datetime(complaints['date_received'])
assert complaints['complaint_id'].is_unique
assert len(complaints) == 400
assert complaints.groupby(['received_year', 'product']).size().eq(20).all()
print(f'Loaded {len(complaints)} public-narrative complaints.')
display(complaints[['complaint_id', 'narrative', 'product',
                    'date_received', 'similarity_group', 'n_words']].head(3))
# Problem 2 code appears under the relevant subparts below.

# Problem 1 - Plan the scikit-learn study (50 points; no AI)

In this problem, you will decide what the study should measure and how it should
be run. You will make those decisions before seeing any model results.

> **Why no AI here?** The exam will ask you to make and defend choices like
> these on your own. This problem gives you a chance to practice with feedback.

## The setting

When someone reports a problem on scikit-learn's GitHub page, a maintainer may
add a label such as `module:metrics` or `module:tree`. These labels help route
the issue to someone with the right expertise. They do not assign a particular
person, and they do not determine whether the issue will be fixed.

The project is considering a tool that recommends one of eight module labels
from the issue title. The current method uses TF-IDF features and multinomial
logistic regression. The proposed method uses a transformer. In either case, a
maintainer makes the final choice.

The eight labels are `module:cluster`, `module:ensemble`,
`module:linear_model`, `module:metrics`, `module:model_selection`,
`module:preprocessing`, `module:tree`, and `module:utils`.

You have a file called `github_issues/issues.csv`. It contains 120 real,
closed scikit-learn issues created between 2018 and 2025: 15 from each of eight
module labels. For each issue, the file gives the issue number, title, creation
date, recorded module label, a pseudonymized reporter group, and a link to the
original issue.

This is a deliberately balanced teaching dataset, not a random sample of all
incoming issues. It leaves out issue bodies, comments, usernames, issues with
more than one module label, and modules outside the selected eight.

a. **What does the recorded label mean? (10 points)**

**(i)** Let $Y_i$ be the module label recorded for issue $i$. In 1-2
sentences, explain what this label tells us. Why is it not proof that the issue
has one objectively correct module label? Give one reason the recorded label
could have been different. (4 points)

**(ii)** Name one group of GitHub issues that `github_issues/issues.csv`
directly describes. Then name one broader group of GitHub issues that the
dataset does not necessarily represent. (3 points)

**(iii)** Imagine that an analyst combines two module labels after discovering
that the new grouping makes the transformer look better. Why is this more than
ordinary data cleaning? (3 points)

### Your response - Problem 1(a)

Write your answer below in the format requested above. You may instead
insert a clear image of handwritten work; if you do, add a one-sentence
typed summary for accessibility.

TODO

b. **Decide what would count as a better model (14 points)**

The study compares three approaches:

1. A **baseline** ignores the issue title and always predicts one module
   label: the most common label in the training data, with alphabetical order
   breaking a tie. This is the **no-information rule** for this study. The full
   120-row dataset is balanced, but the baseline's accuracy on the final test
   set will depend on that test set's label mix.
2. The **simple title model** uses TF-IDF features and multinomial logistic
   regression.
3. The **transformer** is the more complex title model being considered.

The method names are already given; you do not need to copy them into the
table. Instead, plan two checkpoints:

- **Checkpoint 1 - Do titles help at all?** Compare the simple title model with
  the baseline on the same test issues. Plan to report an uncertainty interval
  for the difference. This checkpoint passes if the entire interval is above
  zero.
- **Checkpoint 2 - Is the transformer worth it?** If the simple model helps,
  compare the transformer with the simple model on those same test issues.
  Choose the minimum additional improvement that would justify piloting the
  more complex model. This minimum is the **practical threshold**.


Choose one main performance measure:

- **Accuracy:** the percentage of GitHub issues assigned the recorded module
  label. This is the simplest option and gives every test issue equal weight.
  A module contributes more when it is more common in the chosen test set.
- **Average recall across modules:** compute recall separately for each of the
  eight module labels, then average the eight values. This is also called macro
  recall or balanced accuracy. Use it only if you can explain why equal weight
  for every module fits the intended use.

Complete the table in the notebook. State what the model predicts and how a
maintainer would use it, which scikit-learn GitHub issues the conclusion should
cover, your performance measure, and the rule for passing each checkpoint.
Write both model differences in words; these are the two quantities the study
is designed to estimate.

### Your response - Problem 1(b)

| Question | Your answer |
|---|---|
| What module-label classification does the model make, and how would a maintainer use it? | TODO |
| Which scikit-learn GitHub issues should the conclusion cover? | TODO |
| Main performance measure: accuracy or average recall across modules? Why? | TODO |
| Checkpoint 1: What result for the simple title model versus the baseline would count as evidence that titles help? | TODO |
| Checkpoint 2: How much must the transformer improve over logistic regression before you would recommend a pilot? Why? | TODO |
| Quantity 1 in words: simple title model minus no-information rule | TODO |
| Quantity 2 in words: transformer minus simple title model | TODO |

c. **Choose how to split the 120 GitHub issues (14 points)**

The dataset has 120 rows. Each row is one scikit-learn GitHub issue, and the
model makes one module-label prediction for that issue. The `reporter_group`
column identifies GitHub issues that came from the same pseudonymized reporter.

Use three sets of data:

- **Training set:** fit the models and any text preprocessing.
- **Validation set:** choose model settings and make any allowed design
  choices without using the test results.
- **Test set (final holdout):** compare the two finished models once, after all
  choices have been made.

How you assign GitHub issues to these sets depends on the question you want to
answer. A time-based split can train on older issues and test on newer issues.
A reporter-held-out split keeps each reporter group in only one set and tests
performance on reporters the model has not seen. A random row split assigns
individual GitHub issues at random. It estimates performance for another issue
drawn from roughly the same mix of years and reporters as this dataset. It does
not directly test performance on future issues or on entirely new reporters.

The starter notebook displays the number of issues from each module in each
year. Use that EDA to make sure your proposed training, validation, and test
periods are workable for all eight labels.

Complete the split table in words. You do not need to write code or actually
divide the rows. Work through these decisions:

1. Finish this sentence: “Our final comparison is meant to tell us how the two
   models perform on ______.” Be specific. For example, you might care about
   later scikit-learn issues in the same eight modules or issues submitted by
   reporters not seen during training.
2. Decide whether GitHub issues from the same `reporter_group` must remain in
   the same set. Explain why or why not.
3. Give a concrete rule for assigning the 120 rows to training, validation,
   and test sets. You may use the creation dates, reporter groups, or both.
4. Explain why your rule fits the group of GitHub issues named in step 1. State
   whether it is mainly protecting against changes over time, repeated
   reporters across sets, or both.
5. In one sentence each, explain what a random row split and a
   reporter-held-out split would tell you instead.

Use the final row of the table to explain why no split of
`github_issues/issues.csv` can establish that the model works for a different
software project. The completed table is your full answer; no separate essay
is required.

### Your response - Problem 1(c)

| Decision | Your answer |
|---|---|
| Complete: Our final comparison is meant to tell us how the two models perform on... | TODO |
| Must GitHub issues from the same `reporter_group` stay together? Why or why not? | TODO |
| Which rows go into training, and what happens there? | TODO |
| Which rows go into validation, and what happens there? | TODO |
| Which rows go into the test set, and what happens there? | TODO |
| Does this split mainly address changes over time, repeated reporters, or both? Explain. | TODO |
| What would a random row split tell us instead? | TODO |
| What would a reporter-held-out split tell us instead? | TODO |
| What can `github_issues/issues.csv` not tell us about other software projects? | TODO |

d. **Put the plan together (12 points)**

This is a written planning exercise. Do not write code, fit either model, or
calculate results. Complete the design-brief table in the notebook using words
and your answers above. The brief should contain:

- your research question and hypothesis;
- the separate jobs of the training, validation, and test data;
- an if-then rule that checks both steps: whether the simple title model beats
  the no-information rule, and whether the transformer clears your minimum
  improvement over the simple model; and
- one important conclusion that even a favorable result would not support.

Following the Model Cards reading, also name the tool's intended use and one use
that should be considered out of scope. After the table, write 2-3 sentences
that summarize the plan and cite the relevant part of Section 4 once.

### Your response - Problem 1(d)

| Part of the plan | Your choice |
|---|---|
| Research question / hypothesis | TODO |
| GitHub issues represented in `github_issues/issues.csv` | TODO |
| Issues you want the conclusion to cover | TODO |
| Rule for keeping `reporter_group` together or allowing it across sets | TODO |
| Two comparisons to estimate | TODO |
| Job of the training data | TODO |
| Job of the validation data | TODO |
| Job of the test data | TODO |
| Main measure and the rule for passing each checkpoint | TODO |
| Decision the study will inform | TODO |
| Intended use and intended user | TODO |
| One out-of-scope use | TODO |

**If-then decision rule (1-2 sentences):**  
TODO

**A favorable result would still not establish (1 sentence):**  
TODO

**Model Cards connection:**  
TODO - cite the relevant subsection of Mitchell et al. Section 4

# Problem 2 - Use EDA and run a complaint-routing baseline (30 points)

Now apply the statistical-design workflow to the CFPB complaint-routing
example from Lab 3. You will use EDA, write a small amount of code, and run the
supplied model scaffold.

The question for this problem is fixed:

> Using public complaint narratives from 2022-2024, how well can a simple text
> classifier route public narratives received in 2025 among the five recorded
> product categories?

The file contains 400 public complaint narratives, with 20 complaints from each
product-by-year combination. It also includes a `similarity_group` for repeated
normalized narratives. You will compare a no-information classifier with
TF-IDF multinomial logistic regression. You will not fit a transformer here;
the goal is to build and check the baseline workflow that a later model would
have to beat.

Here is the weak initial workflow. The analyst randomly split rows
from all four years, did not check whether repeated narratives crossed sets,
chose which words to retain using all available narratives, and kept checking
test results while changing settings. In the parts below, replace those four
choices one at a time and use code or output to document the resulting design.

a. **Inspect the data before modeling (4 points)**

Run the supplied profiling code, which prints:

- the number of rows for every product and year;
- the number of `similarity_group` values containing more than one row; and
- the 10th, 50th, and 90th percentiles of `n_words`.

Keep the output visible. In 2-3 sentences, explain what the balanced sampling
means for this exercise and why these public narratives do not represent all
CFPB complaints. Add one assertion that verifies every product-year cell has
20 rows.

In [ ]:
# Inspect the sample before fitting a model.
product_year_counts = pd.crosstab(
    complaints['received_year'], complaints['product']
)
duplicate_group_sizes = complaints.groupby('similarity_group').size()
duplicate_group_count = int((duplicate_group_sizes > 1).sum())
word_count_percentiles = (
    complaints['n_words'].quantile([0.10, 0.50, 0.90]).rename('n_words')
)
display(product_year_counts)
print('Similarity groups with more than one row:', duplicate_group_count)
display(word_count_percentiles.to_frame())
# TODO: add one assertion that every product-year cell contains 20 rows.

### Your response - Problem 2(a)

| Check | Result from your output |
|---|---|
| Rows in each product-year cell | TODO |
| Similarity groups with more than one row | TODO |
| 10th / 50th / 90th percentiles of `n_words` | TODO |

**Interpretation (2-3 sentences):**  
TODO - explain what the balanced sampling means and why public narratives do not represent all CFPB complaints

b. **Build and check the time split (6 points)**

Use the supplied scaffold to create:

- training data from 2022-2023;
- validation data from 2024; and
- test data from 2025.

Add two assertions of your own: one checking the expected number of rows in
each set, and one checking that all five products appear in each set. Run the
provided similarity-group check and preserve its output. Do not fit a model or
look at test performance yet.

In one sentence, explain why the time split matches the stated 2025 question
better than a random row split.

In [ ]:
# Define the split by calendar year before fitting any model.
SPLIT_BY_YEAR = {2022: 'train', 2023: 'train', 2024: 'validation', 2025: 'test'}
complaints_split = complaints.assign(
    split=complaints['received_year'].map(SPLIT_BY_YEAR)
)
assert complaints_split['split'].notna().all()
split_order = ['train', 'validation', 'test']
split_sizes = complaints_split['split'].value_counts().reindex(split_order)
products_by_split = pd.crosstab(
    complaints_split['split'], complaints_split['product']
).reindex(split_order)
group_split_counts = (
    complaints_split[['similarity_group', 'split']].drop_duplicates()
    .groupby('similarity_group')['split'].nunique()
)
cross_split_groups = group_split_counts[group_split_counts > 1]
display(split_sizes.rename('rows').to_frame())
display(products_by_split)
print('Similarity groups appearing in more than one split:', len(cross_split_groups))
# TODO: add an assertion that the split sizes are 200, 100, and 100.
# TODO: add an assertion that all five products appear in every split.
train = complaints_split.query("split == 'train'").copy()
validation = complaints_split.query("split == 'validation'").copy()
test = complaints_split.query("split == 'test'").copy()

### Your response - Problem 2(b)

| Check | Result from your output |
|---|---|
| Training / validation / test rows | TODO |
| Products represented in every split | TODO |
| Similarity groups crossing splits | TODO |

**Why the time split matches the 2025 question (1 sentence):**  
TODO

c. **Fit preprocessing inside the model pipeline (6 points)**

Run the supplied validation loop for `min_df` (**minimum document frequency**)
values 1, 2, and 5. Each candidate must fit `TfidfVectorizer` only on the
training narratives, followed by multinomial logistic regression. Keep the
validation-results table visible and record which `min_df` it selects.

Add one line of code that verifies the selected value came from the candidate
list. Then explain why fitting TF-IDF separately inside each candidate pipeline
prevents the validation and test narratives from influencing which words are
kept.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.pipeline import make_pipeline

CANDIDATE_MIN_DF = [1, 2, 5]

def make_text_model(min_df):
    return make_pipeline(
        TfidfVectorizer(min_df=min_df, ngram_range=(1, 2)),
        LogisticRegression(max_iter=2000, random_state=SEED),
    )

validation_rows = []
for min_df in CANDIDATE_MIN_DF:
    candidate = make_text_model(min_df)
    candidate.fit(train['narrative'], train['product'])
    validation_prediction = candidate.predict(validation['narrative'])
    validation_rows.append({
        'min_df': min_df,
        'validation_accuracy': accuracy_score(
            validation['product'], validation_prediction
        ),
    })
validation_results = pd.DataFrame(validation_rows)
best_min_df = int(
    validation_results.sort_values(
        ['validation_accuracy', 'min_df'], ascending=[False, True]
    ).iloc[0]['min_df']
)
display(validation_results)
print('Selected min_df:', best_min_df)
# TODO: add an assertion that best_min_df is in CANDIDATE_MIN_DF.

### Your response - Problem 2(c)

| `min_df` | Validation accuracy |
|---:|---:|
| 1 | TODO |
| 2 | TODO |
| 5 | TODO |

**Selected value:** TODO  

**Why later narratives cannot influence the vocabulary (1-2 sentences):**  
TODO

d. **Open the test set once (5 points)**

Only after parts (a)-(c) are complete, change `OPEN_TEST` from `False` to
`True`. The supplied code will refit the selected pipeline on the combined
training and validation rows, fit a no-information classifier, and evaluate
both on the 2025 test rows. It also reports a paired bootstrap interval for the
accuracy difference.

Preserve the output. State the two test accuracies, the estimated improvement,
and its interval. Does the interval clear zero? In this complaint-routing
example, that is the same kind of Checkpoint 1 as in Problem 1: does the simple
text model beat the no-information rule?

In [ ]:
from sklearn.dummy import DummyClassifier

OPEN_TEST = False  # Change to True only after parts (a)-(c) are complete.

def paired_bootstrap_accuracy_difference(
    y_true, model_prediction, floor_prediction, B=5000, seed=SEED
):
    y_true = np.asarray(y_true)
    model_prediction = np.asarray(model_prediction)
    floor_prediction = np.asarray(floor_prediction)
    paired_gain = (model_prediction == y_true).astype(float) - (
        floor_prediction == y_true
    ).astype(float)
    rng = np.random.default_rng(seed)
    bootstrap_means = np.array([
        rng.choice(paired_gain, size=len(paired_gain), replace=True).mean()
        for _ in range(B)
    ])
    interval = np.quantile(bootstrap_means, [0.025, 0.975])
    return paired_gain.mean(), interval

if not OPEN_TEST:
    print('Test set remains unopened. Finish parts (a)-(c), then set OPEN_TEST = True.')
else:
    fit_rows = pd.concat([train, validation], ignore_index=True)
    final_model = make_text_model(best_min_df)
    final_model.fit(fit_rows['narrative'], fit_rows['product'])
    no_information = DummyClassifier(strategy='most_frequent')
    no_information.fit(fit_rows[['n_words']], fit_rows['product'])
    model_prediction = final_model.predict(test['narrative'])
    floor_prediction = no_information.predict(test[['n_words']])
    model_accuracy = accuracy_score(test['product'], model_prediction)
    floor_accuracy = accuracy_score(test['product'], floor_prediction)
    improvement, improvement_interval = paired_bootstrap_accuracy_difference(
        test['product'], model_prediction, floor_prediction
    )
    test_results = pd.DataFrame({
        'method': ['No-information classifier', 'TF-IDF logistic regression'],
        'test_accuracy': [floor_accuracy, model_accuracy],
    })
    display(test_results)
    print(f'Paired accuracy improvement: {improvement:.3f}')
    print('Paired bootstrap 95% interval: '
          f'[{improvement_interval[0]:.3f}, {improvement_interval[1]:.3f}]')

### Your response - Problem 2(d)

| Result | Value |
|---|---:|
| No-information test accuracy | TODO |
| TF-IDF logistic-regression test accuracy | TODO |
| Paired accuracy improvement | TODO |
| Paired bootstrap 95% interval | TODO |

**Checkpoint 1 conclusion (1-2 sentences):**  
TODO - say whether the interval clears zero and what that means

e. **Document the design changes (4 points)**

Complete the four-row design table in the notebook. For each row, name the
problem in the initial workflow, state the design decision, and point to the
supporting code or output: time split, similarity-group check, TF-IDF fitting,
and one-time test evaluation.

Finish with two sentences: one limitation reduced by the code and one that the
code cannot solve. The remaining limitation must discuss selection into the
public-narrative dataset.

### Your response - Problem 2(e)

| Design decision | Problem in the initial workflow | Supporting code or output |
|---|---|---|
| Time split | TODO | TODO |
| Similarity-group check | TODO | TODO |
| TF-IDF inside the model pipeline | TODO | TODO |
| One-time test evaluation | TODO | TODO |

**Limitation reduced by the code (1 sentence):**  
TODO

**Limitation the code cannot solve (1 sentence):**  
TODO - discuss selection into the public-narrative dataset

f. **AI stretch (required): simulate duplicate contamination (5 points)**

Before using AI, predict which split will look better and why. Then use AI to
help write a small, reproducible simulation with repeated groups and a
group-specific fingerprint. Compare the same simple classifier under:

1. a random row split that can place rows from one group in both training and
   test data; and
2. a group-held-out split with no group shared across the boundary.

Use a fixed seed, preserve the final code and output, and verify the group
overlap for both splits. Report both accuracies and explain whether the result
matched your prediction. Finally, connect the simulation to
`similarity_group` in the complaint data and name one way the simulation is
less realistic than actual complaint narratives.

Record the initial AI prompt and your checks in the AI-use table. Full credit
does not require the random-row result to be larger; a surprising or failed
result can earn full credit when the code and diagnosis are accurate.

In [ ]:
# Stretch workspace: paste or adapt the AI-assisted simulation below.
# Keep a fixed seed and show the group overlap and accuracy for both splits.
STRETCH_SEED = SEED
# TODO: add and run your final simulation code in this cell or new cells below.

### Your response - Problem 2(f)

**Prediction recorded before using AI:**  
TODO

| Split | Groups shared by training and test | Accuracy |
|---|---:|---:|
| Random row split | TODO | TODO |
| Group-held-out split | TODO | TODO |

**What happened and why (2-3 sentences):**  
TODO

**Connection to complaint `similarity_group` (1-2 sentences):**  
TODO

**One way the simulation is less realistic (1 sentence):**  
TODO

# Problem 3 - See what a detailed prompt changes (20 points)

Problem 2 gave you a worked complaint-routing example. Now return to the
scikit-learn study you designed in Problem 1. You will ask the same AI tool and
model for two analysis plans. Start a fresh conversation for each request so
the first answer cannot influence the second. Give both conversations the same
`issues.csv` file, or the same description of the data and its columns.

- **Original request:** “Use the supplied scikit-learn issues to determine
  whether the transformer is better than logistic regression for issue
  triage.” Ask the assistant to propose an analysis plan.
- **Detailed request:** Give the assistant your completed design brief from
  Problem 1 and ask for an analysis plan that could be carried out as written.

You will receive only one answer to each prompt. That is enough to compare the
two answers, but not enough to prove that every difference was caused by the
wording of the prompt. AI systems can give different answers even when asked
the same question twice.

a. **Compare the two answers (8 points)**

Complete the five-row table in the notebook. Compare what each answer says
about:

- the target population and outcome;
- the unit of analysis and data split;
- the comparison ladder: no-information rule, simple model, and transformer;
- uncertainty and the size of an improvement that would matter; and
- the conclusion and decision the evidence could support.

For each row, quote or point to specific wording in the response. If something
important is missing, say so directly.

### Your response - Problem 3(a)

| What to compare | Original-request answer: evidence or omission | Detailed-request answer: evidence or omission | What you conclude |
|---|---|---|---|
| Target population and outcome | TODO | TODO | TODO |
| Unit and split | TODO | TODO | TODO |
| No-information rule, simple model, and transformer comparison | TODO | TODO | TODO |
| Uncertainty and practical importance | TODO | TODO | TODO |
| Supported claim and decision | TODO | TODO | TODO |

b. **What improved, and what did not? (5 points)**

Describe one meaningful improvement in the answer to the detailed request.
Then describe one error or omission that remains. Use 1-2 sentences for each
and point to evidence in the responses.

Add one sentence explaining why two answers are not enough to separate the
effect of the prompt from ordinary variation in the model's responses.

### Your response - Problem 3(b)

Write your answer below in the format requested above. You may instead
insert a clear image of handwritten work; if you do, add a one-sentence
typed summary for accessibility.

TODO

c. **Make the final call yourself (5 points)**

Write the corrections an analyst would need before carrying out either plan.
Do not ask the AI for a third rewrite. End by naming one study-design decision
that remained your responsibility, even if the assistant happened to recommend
the same choice.

## Required AI-use record (2 points)

Complete the AI-use table in the notebook. Record the initial prompt for each
use, but not the full conversation. Paste the two complete first responses in
**Appendix A: Raw AI evidence**. Do not rely on an external link.

| Assignment part | Tool | Purpose | Initial prompt only | What I checked | What changed after checking | Decision I remained responsible for |
|---|---|---|---|---|---|---|
| Problem 2(f) | TODO | Build the exploratory grouped-split simulation | TODO | TODO | TODO | TODO |
| Problem 3 original request | TODO | Obtain an analysis plan from the original request | TODO | TODO | TODO | TODO |
| Problem 3 detailed request | TODO | Obtain a plan based on the completed design brief | TODO | TODO | TODO | TODO |

### Your response - Problem 3(c)

Write your answer below in the format requested above. You may instead
insert a clear image of handwritten work; if you do, add a one-sentence
typed summary for accessibility.

TODO

## Appendix A: Raw AI evidence

Paste the two complete **first** plan outputs below. Preserve the wording
exactly as returned. Do not include follow-up conversation turns.

### A1. Weak-request plan output

TODO - paste the complete first response here

### A2. Specified-request plan output

TODO - paste the complete first response here

**Appendix check:** Both entries contain the complete first response and no
follow-up turns. The same tool/model and data context were used for both.

## Final submission check

- [ ] I restarted the runtime and ran all cells from top to bottom.
- [ ] Every requested denominator, table, figure, excerpt, and interpretation is visible.
- [ ] Any handwritten images are legible and each has a typed description or statistical conclusion.
- [ ] Required raw AI prompts/outputs and the AI-use record are preserved.
- [ ] I opened my downloaded PDF and `.ipynb` before uploading them.

The PDF is the primary grading surface; the notebook is the executable record. Both represent the same work.